# Train the GPU labs on Colab → publish everything to Hugging Face

Run this on a **GPU runtime** (Runtime → Change runtime type → **T4 GPU**). It:
1. clones Ropedia Academy (which already ships the CPU-trained bundles in `models/`),
2. trains the labs that benefit from a GPU here (NeRF; add more below),
3. publishes **all** bundles to your account and one **Collection** — one write token covers every repo.

> Get a write token at https://huggingface.co/settings/tokens (role: Write).

In [ ]:
!git clone -q https://github.com/ChaoYue0307/ropedia-academy.git
%cd ropedia-academy
!pip -q install huggingface_hub
import torch
print("GPU available:", torch.cuda.is_available())
from huggingface_hub import notebook_login
notebook_login()   # paste your WRITE token (one token covers all repos)

## 1 · Train the GPU-worthy labs
Each call runs that lab's own (verified) code on the GPU and writes `outputs/<lab>/`. NeRF is included; uncomment others as you like.

In [ ]:
import os, json, shutil

def run_lab(nb, steps):
    os.environ["STEPS"] = str(steps)
    cells = [c for c in json.load(open(nb))["cells"] if c["cell_type"] == "code"]
    src = "\n".join("".join(c["source"]) for c in cells if "notebook_login" not in "".join(c["source"]))
    src = "\n".join(l for l in src.split("\n") if not l.strip().startswith(("!", "%")))
    exec(src, {"__name__": "__main__"})
    print("trained:", nb)

run_lab("notebooks/training/B_nerf_from_scratch.ipynb", 3000)   # ~a few min on a T4
# run_lab("notebooks/training/B_hashgrid_instngp.ipynb", 5000)
# run_lab("notebooks/training/A_motion_diffusion.ipynb", 6000)

os.makedirs("models", exist_ok=True)
for d in os.listdir("outputs"):
    p = os.path.join("outputs", d)
    if os.path.isdir(p):
        shutil.copytree(p, os.path.join("models", d), dirs_exist_ok=True)
print("bundles ready:", sorted(os.listdir("models")))

## 2 · Publish everything to your account + one Collection
Uploads every `models/<lab>/` to `<you>/ropedia-<lab>` (public by default) and gathers them into the *Ropedia Academy — trained models* collection.

In [ ]:
!python scripts/upload_all_to_hf.py
# options:  --private   --org <name>   --no-collection

## Foundation fine-tunes (QLoRA · VideoMAE · DINOv2 · Whisper · SD-LoRA · …)
Those need their own installs / datasets, so run them from the **Labs** tab one at a time: open the lab → **Run all** → its *Publish to the Hugging Face Hub* cell. They'll land in the same account and you can add them to the collection too.